# Pokédex AI — treino no Google Colab (GPU)

Treina e compara as arquiteturas no dataset de Pokémon da Gen 1.

**Antes de começar:** menu **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4**.

Depois rode as células em ordem (Shift+Enter). Os modelos são salvos no seu Google Drive em
`MyDrive/pokedex-ai/models/`, então nada se perde se o Colab desconectar.

## 1. Conferir a GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "Sem GPU! Troque o ambiente de execução para GPU T4."
print("PyTorch", torch.__version__, "· GPU:", torch.cuda.get_device_name(0))

## 2. Google Drive (onde os modelos ficam salvos)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

OUT_DIR = "/content/drive/MyDrive/pokedex-ai/models"
!mkdir -p {OUT_DIR}

## 3. Código do projeto

In [ ]:
%cd /content
!rm -rf pokedex-ai
!git clone -q https://github.com/raphaelcomph1/Classificador_de_Pok-mon.git pokedex-ai
%cd /content/pokedex-ai
!git log --oneline -1
# torch, torchvision, scikit-learn etc. já vêm no Colab; instala só o que falta
!pip install -q -e .

## 4. Dados: download, checagem e split

O split é determinístico (seed 42 e arquivos ordenados), então sai **idêntico** ao feito localmente.
A célula confere isso pelas contagens. Leva alguns minutos (a checagem calcula hash de ~6,8 mil imagens).

In [ ]:
import os
os.environ["POKEDEX_DATA_ROOT"] = "/content/data"

!python -m pokedex.data.download
!python -m pokedex.data.check
!python -m pokedex.data.split

import csv
from collections import Counter
with open("/content/data/processed/splits.csv", encoding="utf-8") as f:
    counts = Counter(row["split"] for row in csv.DictReader(f))
print(counts)
assert counts == {"train": 4728, "val": 1016, "test": 1016}, "Split diferente do local!"
print("Split idêntico ao local ✔")

## 5. Treino

Cada arquitetura treina com a mesma seed, o mesmo split e as mesmas transforms (comparação justa).
Tempo aproximado numa T4: `small_cnn` ~5 min, `mobilenet_v3_large` ~8 min, `resnet18` ~8 min.

Para treinar só uma, rode a célula dela. Se o Colab desconectar, rode de novo a partir da seção 1:
o que já terminou continua salvo no Drive.

In [ ]:
# Baseline treinado do zero (régua de comparação)
!python -m pokedex.train --arch small_cnn --num-workers 2 --output-dir {OUT_DIR}

In [ ]:
!python -m pokedex.train --arch mobilenet_v3_large --num-workers 2 --output-dir {OUT_DIR}

In [ ]:
!python -m pokedex.train --arch resnet18 --num-workers 2 --output-dir {OUT_DIR}

In [ ]:
# (Opcional) mais pesada: ~15 min. Rode se sobrar tempo de GPU.
# !python -m pokedex.train --arch resnet50 --num-workers 2 --output-dir {OUT_DIR}

## 6. Avaliação no conjunto de teste

O teste é usado **uma única vez**, depois de todo o treino. Gera `metrics_test.json` e
`confusion_test.png` ao lado de cada `best.pt`.

In [ ]:
from pathlib import Path

ARCHS = ["small_cnn", "mobilenet_v3_large", "resnet18", "resnet50"]
trained = [a for a in ARCHS if Path(OUT_DIR, a, "best.pt").exists()]
print("Treinados:", trained)
for arch in trained:
    !python -m pokedex.evaluate {OUT_DIR}/{arch}/best.pt --num-workers 2
    print()

## 7. Comparação

In [ ]:
import json
import pandas as pd

rows = []
for arch in trained:
    hist = json.loads(Path(OUT_DIR, arch, "history.json").read_text())
    test = json.loads(Path(OUT_DIR, arch, "metrics_test.json").read_text())
    rows.append({
        "arquitetura": arch,
        "val_acc (melhor)": hist["best_val_acc"],
        "test_acc": test["accuracy"],
        "test_top3": test["top3_accuracy"],
        "test_f1_macro": test["macro_f1"],
        "minutos": hist["minutes"],
    })
summary = pd.DataFrame(rows).sort_values("test_acc", ascending=False)
summary.style.format({c: "{:.3f}" for c in summary.columns if c not in ("arquitetura", "minutos")})

In [ ]:
# Curvas de treino: acurácia de treino x validação por época
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(trained), figsize=(5 * len(trained), 4), squeeze=False)
for ax, arch in zip(axes[0], trained):
    hist = pd.DataFrame(json.loads(Path(OUT_DIR, arch, "history.json").read_text())["history"])
    x = range(1, len(hist) + 1)
    ax.plot(x, hist["train_acc"], label="treino")
    ax.plot(x, hist["val_acc"], label="validação")
    if (hist["stage"] == "finetune").any():
        ax.axvline((hist["stage"] == "head").sum() + 0.5, color="gray", ls="--", label="início fine-tuning")
    ax.set_title(arch); ax.set_xlabel("época"); ax.set_ylim(0, 1); ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/curvas_treino.png", dpi=120)
plt.show()

In [ ]:
# Classes mais confundidas e piores classes do melhor modelo
best_arch = summary.iloc[0]["arquitetura"]
test = json.loads(Path(OUT_DIR, best_arch, "metrics_test.json").read_text())
print(f"Melhor modelo: {best_arch}\n")
print("Pares mais confundidos (verdadeiro -> previsto):")
for c in test["most_confused"][:10]:
    print(f"  {c['true']:>18} -> {c['predicted']:<18} {c['count']}x")
print("\nPiores classes (F1):")
for name, f1 in test["worst_classes"]:
    print(f"  {name:>18}  {f1:.2f}")

from IPython.display import Image, display
display(Image(f"{OUT_DIR}/{best_arch}/confusion_test.png", width=900))

## 8. Levar os resultados de volta

Os arquivos ficam no Drive em `MyDrive/pokedex-ai/models/`. A célula abaixo junta **só as métricas**
(arquivos pequenos, sem os pesos) num zip para baixar. Copie o zip para o projeto local, para
documentar os resultados em `docs/decisoes.md`.

Para usar o modelo na API/app (Fases 3 e 4), baixe também o `best.pt` do melhor modelo pelo Drive
e coloque em `models/<arquitetura>/best.pt` no projeto local.

In [ ]:
import shutil, tempfile

tmp = Path(tempfile.mkdtemp()) / "resultados"
for arch in trained:
    for name in ("history.json", "metrics_test.json", "confusion_test.png"):
        src = Path(OUT_DIR, arch, name)
        if src.exists():
            (tmp / arch).mkdir(parents=True, exist_ok=True)
            shutil.copy(src, tmp / arch / name)
shutil.copy(f"{OUT_DIR}/curvas_treino.png", tmp)
zip_path = shutil.make_archive(f"{OUT_DIR}/../resultados", "zip", tmp)
print("Zip:", zip_path)

from google.colab import files
files.download(zip_path)